# Code Demonstration

#### Necesarry Libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# !pip install matplotlib numpy
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageFilter
from IPython.display import display
import cv2
from ultralytics import YOLO
import os
import random
import PIL.ImageChops as ImageChops

## YOLO enemy detection

In [ ]:
# File paths to different assets
file_enemy = "enemy"
file_env = "environment"

In [ ]:
def preprocess_frame(frame):
    # Convert frame from BGR (OpenCV default) to RGB
    frame_np = np.array(frame)
    frame_rgb = cv2.cvtColor(frame_np, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)
    
    # First crop: according to processing code (assumed frame is 2560x1440)
    crop_box_1 = (0, 0, 2080, 1440)
    image_cropped = pil_image.crop(crop_box_1)
    
    # Second crop: crop equally from left and right to obtain a 1440x1440 square
    crop_box_2 = (320, 0, 320 + 1440, 1440)
    final_image = image_cropped.crop(crop_box_2)
    
    # Resize to 640x640 (model input size)
    img_resized = final_image.resize((640, 640))
    
    # Convert back to BGR numpy array for OpenCV
    processed_frame = cv2.cvtColor(np.array(img_resized), cv2.COLOR_RGB2BGR)
    return processed_frame

def apply_filter_2(image, param1):
    """
    Applies a Sobel edge detector.
    'param1' is used as the kernel size.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Compute gradients along the x and y axis, using kernel size (must be odd)
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=param1)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=param1)
    sobel = cv2.magnitude(sobelx, sobely)
    # Normalize the result to 0-255 then convert back to uint8
    sobel = np.uint8(255 * sobel / np.max(sobel))
    return cv2.cvtColor(sobel, cv2.COLOR_GRAY2BGR)

def add_outline(img, thickness=1, outline_color=(0, 0, 0, 255)):
    """
    Adds a crisp black outline to a transparent image by dilating the alpha channel.
    Uses an expansion approach to ensure the outline sits around the enemy.

    Parameters:
      img: PIL.Image object in RGBA mode.
      thickness: outline thickness in pixels.
      outline_color: RGBA tuple for the outline color.

    Returns:
      A new PIL.Image with an outline added.
    """
    if thickness <= 0:
        return img

    # Ensure the image is in RGBA.
    img = img.convert("RGBA")

    # Create a new blank image to accommodate the outline
    new_size = (img.width + thickness * 2, img.height + thickness * 2)
    canvas = Image.new("RGBA", new_size, (0, 0, 0, 0))
    canvas.paste(img, (thickness, thickness))

    # Extract and dilate the alpha channel
    alpha = canvas.split()[-1]
    dilated = alpha.filter(ImageFilter.MaxFilter(thickness * 2 + 1))

    # The outline mask is the dilated alpha minus the original alpha
    outline_mask = ImageChops.subtract(dilated, alpha)

    # Create an image for the outline using the outline_color
    outline_img = Image.new("RGBA", new_size, (0, 0, 0, 0))
    outline_img.paste(outline_color, mask=outline_mask)

    # Composite the outline and the original image
    final_img = Image.alpha_composite(outline_img, canvas)
    return final_img


def boxes_overlap(box1, box2):
    return not (box1[2] <= box2[0] or box1[0] >= box2[2] or 
                box1[3] <= box2[1] or box1[1] >= box2[3])

def is_mostly_transparent(img, threshold=0.5):
    if img.mode != "RGBA":
        img = img.convert("RGBA")
    alpha = img.getchannel("A")
    pixels = list(alpha.getdata())
    transparent_count = sum(1 for px in pixels if px == 0)
    return (transparent_count / len(pixels)) > threshold

def crop_enemy(enemy_img):
    if random.random() < 0.3:  # 30% chance to crop
        direction = random.choice(["left", "right", "top", "bottom"])
        crop_ratio = random.uniform(0, 0.3)  # Crop between 0-30%
        width, height = enemy_img.size

        if direction == "left":
            enemy_img = enemy_img.crop((int(width * crop_ratio), 0, width, height))
        elif direction == "right":
            enemy_img = enemy_img.crop((0, 0, int(width * (1 - crop_ratio)), height))
        elif direction == "top":
            enemy_img = enemy_img.crop((0, int(height * crop_ratio), width, height))
        elif direction == "bottom":
            enemy_img = enemy_img.crop((0, 0, width, int(height * (1 - crop_ratio))))
    return enemy_img

def generate_yolo_sample(env_img_path, enemy_directory, output_directory=None, val_split=0.2):
    enemy_files = [f for f in os.listdir(enemy_directory) if f.endswith('.png')]
    if not enemy_files:
        print("No enemy images found.")
        return None, None

    # Open and preprocess environment image
    env_img = Image.open(env_img_path).convert("RGBA")
    env_np = preprocess_frame(env_img)
    env_rgb = cv2.cvtColor(env_np, cv2.COLOR_BGR2RGB)
    env_img = Image.fromarray(env_rgb)
    env_width, env_height = env_img.size

    restricted_areas = [
        (0, 0, 76, 144),    # Top-left restricted area
        (300, 287, 336, 326)
    ]

    annotations = []
    placed_boxes = []
    n_enemies = random.randint(1, 4)
    enemy_count = 0

    while enemy_count < n_enemies:
        enemy_file = random.choice(enemy_files)
        enemy_path = os.path.join(enemy_directory, enemy_file)
        enemy_img = Image.open(enemy_path).convert("RGBA")

        if is_mostly_transparent(enemy_img, 0.5):
            continue

        enemy_img = crop_enemy(enemy_img)
        enemy_img_resized = enemy_img.resize((enemy_img.width * 2, enemy_img.height * 2), resample=Image.BICUBIC)
        enemy_img_resized = add_outline(enemy_img_resized, thickness=1)
        enemy_img_resized = enemy_img_resized.resize((enemy_img_resized.width * 2, enemy_img_resized.height * 2), resample=Image.BOX)
        e_width, e_height = enemy_img_resized.size

        max_x = env_width - e_width
        max_y = env_height - e_height
        if max_x <= 0 or max_y <= 0:
            continue

        rand_x = random.randint(0, max_x)
        rand_y = random.randint(0, max_y)
        new_box = (rand_x, rand_y, rand_x + e_width, rand_y + e_height)

        if any(boxes_overlap(new_box, placed) for placed in placed_boxes):
            continue
        if any(boxes_overlap(new_box, restricted) for restricted in restricted_areas):
            continue

        env_img.paste(enemy_img_resized, (rand_x, rand_y), enemy_img_resized)
        placed_boxes.append(new_box)

        x_center = (rand_x + e_width / 2) / env_width
        y_center = (rand_y + e_height / 2) / env_height
        norm_width = e_width / env_width
        norm_height = e_height / env_height
        annotations.append(f"0 {x_center:.6f} {y_center:.6f} {norm_width:.6f} {norm_height:.6f}")

        enemy_count += 1

    # Apply Sobel filtering
    env_img_rgb = env_img.convert("RGB")
    env_np = np.array(env_img_rgb)
    env_cv = cv2.cvtColor(env_np, cv2.COLOR_RGB2BGR)
    filtered_env_cv = apply_filter_2(env_cv, param1=3)
    filtered_env_rgb = cv2.cvtColor(filtered_env_cv, cv2.COLOR_BGR2RGB)
    filtered_env_pil = Image.fromarray(filtered_env_rgb)

    # Optionally save output if output_directory is provided
    if output_directory:
        output_subdir = os.path.join(output_directory, 'val')
        os.makedirs(output_subdir, exist_ok=True)

        base_filename = os.path.splitext(os.path.basename(env_img_path))[0]
        output_img_path = os.path.join(output_subdir, base_filename + '.png')

        filtered_env_pil.save(output_img_path)

        annotation_path = output_img_path.replace('.png', '.txt')
        with open(annotation_path, 'w') as f:
            f.write("\n".join(annotations))

    return filtered_env_pil, annotations

In [ ]:
filtered_img, annotations = generate_yolo_sample(
    env_img_path="example_image.png",
    enemy_directory="example_enemies"
)

display(filtered_img)
print("\n".join(annotations))

# Load a model
model = YOLO("knut_sandbox/runs/detect/train/weights/best.pt")

results = model(filtered_img, conf=0.6)  

annotated_bgr = results[0].plot()
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
display(Image.fromarray(annotated_rgb))



## Projectile Tracker

### Imports and Configuration

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML, display
import base64
import os

from moviepy.editor import VideoFileClip
from IPython.display import HTML
# Notebook-friendly defaults
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)

# --- Config flags (toggle interactively later if you like) ---
DEBUG             = False   # turn on extra drawings
WRITE_VIDEO       = True    # final MP4
DOWNSCALE_FACTOR  = 2       # must match gaussian_downsize
VID_PATH          = "projectile_tracking/120fps_data/movement_green.mp4"
OUT_DIR           = Path("projectile_tracking/optical_flow")
OUT_DIR.mkdir(exist_ok=True, parents=True)

### Utility Functions

In [ ]:
# === Projectile-tracking utility functions =================================


# ---------------------------------------------------------------------------
# video-preparation helpers
# ---------------------------------------------------------------------------


def gaussian_downsize(frame, ksize: int = 19, sigma: float = 1.2,
                      factor: int = DOWNSCALE_FACTOR):
    """Blur, then shrink the frame by <factor> to cut computation in half."""
    blur = cv.GaussianBlur(frame, (ksize, ksize), sigma)
    h, w = blur.shape[:2]
    return cv.resize(blur, (w // factor, h // factor), interpolation=cv.INTER_AREA)






def show_video_segment(path, start_frame=None, end_frame=None, fps_override=None, width=720):
    """
    Display a video segment inline without manual ffmpeg tinkering.
    """
    clip      = VideoFileClip(path)
    clip_fps = clip.fps
    fps_override = clip_fps if fps_override is None else fps_override
    fps       = fps_override
    t_start   = (start_frame or 0)   / clip.fps
    t_end     = (end_frame   or clip.duration*clip.fps) / clip.fps
    sub_clip  = clip.subclip(t_start, t_end)

    # MoviePy’s built-in helper returns an HTML5 <video> tag
    return sub_clip.ipython_display(width=width, autoplay=False, loop=False, maxduration=60)

# ---------------------------------------------------------------------- #
#  Helper utilities                                                      #
# ---------------------------------------------------------------------- #



# ---------------------------------------------------------------------------
# optical-flow helpers
# ---------------------------------------------------------------------------
def compute_optical_flow(prev_gray, current_gray, **kwargs):
    """Thin wrapper around Farnebäck with sensible defaults, override by kwargs."""
    defaults = dict(pyr_scale=0.8, levels=3, winsize=15,
                    iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
    defaults.update(kwargs)
    return cv.calcOpticalFlowFarneback(prev_gray, current_gray, None, **defaults)


def identify_background_motion(magnitude, angle, dominant_angle_deg, block_rows=6, block_cols=12):
        """Estimate background motion magnitude block-wise in dominant direction."""
        h, w = magnitude.shape
        block_h = h // block_rows
        block_w = w // block_cols

        # Compute angular deviation from the dominant background angle
        angle_deviation = np.abs(angle - dominant_angle_deg)
        angle_deviation = np.minimum(angle_deviation, 360 - angle_deviation)
        background_mask = (angle_deviation <= 10)  # ±10° window

        block_maxes = []

        for i in range(0, h, block_h):
            for j in range(0, w, block_w):
                y1, y2 = i, min(i + block_h, h)
                x1, x2 = j, min(j + block_w, w)

                block_mag = magnitude[y1:y2, x1:x2]
                block_mask = background_mask[y1:y2, x1:x2]

                masked_mag = block_mag[block_mask]
                if masked_mag.size > 0:
                    block_max = np.max(masked_mag)
                    block_maxes.append(block_max)


        bg_magnitude = np.mean(sorted(block_maxes))
        return  bg_magnitude


def subtract_background(flow):
        # 1) Convert flow to magnitude and angle (in degrees)
        magnitude, angle = cv.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)
        
        if np.mean(magnitude) <= 1:
            # If too low mean magnitude the background is not moving, return untouched flow vector.
            return flow[..., 0], flow[..., 1], (0, 0)

        # 2) Create overlapping bins (every 5 degrees)
        angle_shifted = (angle + 5) % 360  # Shift by half-bin to center the bins
        angle_bin = (angle_shifted // 10).astype(np.uint8)  # 0 to 71 bins

        # 3) Find dominant angle bin
        dominant_bin = np.bincount(angle_bin.flatten()).argmax()
        dominant_angle_deg = dominant_bin * 10  # Central angle of dominant bin
        
        # 4) Calculate background motion estimate for projectile tracking and prediction.
        background_mag = identify_background_motion(magnitude, angle, dominant_angle_deg)
        
        # Convert angle to radians
        dominant_angle_rad = np.deg2rad(dominant_angle_deg)

        # Convert polar to Cartesian
        background_vx = background_mag * np.cos(dominant_angle_rad)
        background_vy = background_mag * np.sin(dominant_angle_rad)

        # 4) Calculate angular deviation
        angle_deviation = np.abs(angle - dominant_angle_deg)
        angle_deviation = np.minimum(angle_deviation, 360 - angle_deviation)

        # 5) Find background pixels
        background_mask = (angle_deviation <= 10)  # ±5 degrees window

        # 6) Background magnitude profile
        background_magnitudes = magnitude[background_mask]
        if len(background_magnitudes) == 0:
            background_mag_threshold = 0.0
        else:
            background_mag_threshold = np.percentile(background_magnitudes, 90)

        # 7) Suppression
        suppress_mask = (background_mask) & (magnitude <= background_mag_threshold * 1.5)

        # 8) Subtract suppressed flow
        residual_flow_x = flow[..., 0].copy()
        residual_flow_y = flow[..., 1].copy()

        residual_flow_x[suppress_mask] = 0
        residual_flow_y[suppress_mask] = 0

        return residual_flow_x, residual_flow_y, (background_vx, background_vy)


# ---------------------------------------------------------------------------
# motion-cluster helpers
# ---------------------------------------------------------------------------
def bins_are_neighbors(bin1, bin2, num_bins):
    """Treat angle bins as circular and ask if they differ by ≤2."""
    diff = abs(bin1 - bin2)
    return diff <= 2 or diff >= (num_bins - 2)


def boxes_overlap(boxA, boxB, margin: int = 5):
    """Axis-aligned intersection-test with a small tolerance."""
    Ax1, Ay1, Ax2, Ay2 = boxA
    Bx1, By1, Bx2, By2 = boxB
    if Ax2 + margin < Bx1 or Bx2 + margin < Ax1:
        return False
    if Ay2 + margin < By1 or By2 + margin < Ay1:
        return False
    return True


def merge_boxes(boxA, boxB):
    """Return the union of two bounding boxes."""
    Ax1, Ay1, Ax2, Ay2 = boxA
    Bx1, By1, Bx2, By2 = boxB
    return (min(Ax1, Bx1), min(Ay1, By1), max(Ax2, Bx2), max(Ay2, By2))


def pixel_clustering(res_x, res_y, bg_velocity=(0, 0), mag_threshold: float = 0.5,
                     size_threshold: int = 10, num_bins: int = 36):
    """
    Connected-component clustering of strong-motion pixels.
    Returns [(bin_id, (x1,y1,x2,y2), pixel_list), ...]
    """
    res_x -= bg_velocity[0]
    res_y -= bg_velocity[1]
    mag, ang = cv.cartToPolar(res_x, res_y, angleInDegrees=False)
    strong = mag > mag_threshold
    ang_deg = np.degrees(ang)
    ang_bin = (ang_deg // (360 / num_bins)).astype(np.uint8)

    bboxes = []
    for bin in range(num_bins):
        mask_b = ((ang_bin == bin) & strong).astype(np.uint8)
        if mask_b.sum() == 0:
            continue
        n_labels, labels = cv.connectedComponents(mask_b, connectivity=8)
        for lbl in range(1, n_labels):
            ys, xs = np.where(labels == lbl)
            if len(xs) < size_threshold:
                continue
            x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
            pixels = list(zip(xs, ys))         # (x, y)
            bboxes.append((bin, (x1, y1, x2, y2), pixels))
    return bboxes


def filter_and_merge_bounding_boxes(bounding_boxes, min_l: int, max_l: int,
                                    num_bins: int = 36):
    """Drop outliers by size and iteratively merge overlaps in neighbouring bins."""
    keep = []
    for bin, (x1, y1, x2, y2), pix in bounding_boxes:
        w, h = x2 - x1, y2 - y1
        if min_l <= w <= max_l and min_l <= h <= max_l:
            keep.append((bin, (x1, y1, x2, y2), pix))

    merged = True
    while merged:
        merged = False
        result = []
        while keep:
            cb, cbox, cpix = keep.pop()
            merged_idx = None
            for i, (rb, rbox, rpix) in enumerate(result):
                if bins_are_neighbors(rb, cb, num_bins) and boxes_overlap(cbox, rbox):
                    new_box = merge_boxes(cbox, rbox)
                    result[i] = (rb, new_box, cpix + rpix)
                    merged = True
                    merged_idx = i
                    break
            if merged_idx is None:
                result.append((cb, cbox, cpix))
        keep = result
    return keep


# ---------------------------------------------------------------------------
# simple geometry helpers
# ---------------------------------------------------------------------------
def centre(bbox):
    x1, y1, x2, y2 = bbox
    return (x1 + x2) / 2, (y1 + y2) / 2


def estimate_flow(pixel_coords, flow_x, flow_y):
    """Average flow vector over an arbitrary pixel list."""
    if not pixel_coords:
        return 0.0, 0.0
    vx = [flow_x[y, x] for (x, y) in pixel_coords]
    vy = [flow_y[y, x] for (x, y) in pixel_coords]
    return float(np.mean(vx)), float(np.mean(vy))


def create_center_mask(frame_shape, mask_size_ratio: float = 0.2,
                       anchor_point: tuple | None = None):
    """
    Boolean mask that zeroes out a square in the middle (or around anchor_point).
    Returns (mask, (x1,y1,x2,y2)).
    """
    h, w = frame_shape[:2]
    mask = np.ones((h, w), np.uint8)
    side = int(h * mask_size_ratio)
    half = side // 2
    cx, cy = (w // 2, h // 2) if anchor_point is None else anchor_point
    cx, cy = np.clip(cx, 0, w - 1), np.clip(cy, 0, h - 1)
    x1, x2 = max(cx - half, 0), min(cx + half, w)
    y1, y2 = max(cy - half, 0), min(cy + half, h)
    mask[y1:y2, x1:x2] = 0
    return mask.astype(bool), (x1, y1, x2, y2)


class Projectile:
    def __init__(self, bbox, bin_id, flow_vector):
        self.bbox = bbox
        self.bin_id = bin_id
        self.global_velocity_log = [flow_vector]  #Expects bg_velocity subtracted.
        self.center = centre(bbox)
        self.id = None
        self.age = 0
        self.missed = 0
        self.confirmed = False
    

    
def update_projectiles(projectiles: list[Projectile], detections, res_x, res_y, next_id=0, num_bins=36, max_missed=3, bg_velocity: tuple = (0, 0)):
    predictions = []
    for projectile in projectiles:
        cx, cy = centre(projectile.bbox)
        vx, vy = projectile.global_velocity_log[-1]
        pred_cx = cx + vx + bg_velocity[0]
        pred_cy = cy + vy + bg_velocity[1]
        predictions.append((projectile, (pred_cx, pred_cy)))

    assigned_projectiles = set()
    assigned_detections = set()

    for i, (bin_id, (x1, y1, x2, y2), pixel_coords) in enumerate(detections):
        cx_det, cy_det = centre((x1, y1, x2, y2))
        best_projectile = None
        best_distance = float('inf')

        for projectile, (pred_cx, pred_cy) in predictions:
            dist = np.hypot(pred_cx - cx_det, pred_cy - cy_det)
            bin_diff = min(abs(projectile.bin_id - bin_id), num_bins - abs(projectile.bin_id - bin_id))

            if dist < 30 and bin_diff <= 2:
                if dist < best_distance:
                    best_distance = dist
                    best_projectile:Projectile = projectile

        if best_projectile is not None:
            # Update existing projectile
            best_projectile.bbox = (x1, y1, x2, y2)
            best_projectile.center = centre((x1, y1, x2, y2))
            mean_vx, mean_vy = estimate_flow(pixel_coords, res_x, res_y)
            best_projectile.global_velocity_log.append((mean_vx - bg_velocity[0], mean_vy - bg_velocity[1]))
            if len(best_projectile.global_velocity_log) > 10:
                best_projectile.global_velocity_log.pop(0)
            best_projectile.bin_id = bin_id
            best_projectile.age += 1
            
            if best_projectile.age >= 3 and not best_projectile.confirmed:
                best_projectile.confirmed = True
                best_projectile.id = next_id
                next_id += 1
                
            best_projectile.missed = 0
            assigned_projectiles.add(best_projectile)
            assigned_detections.add(i)

    # Create new tracks for unmatched detections
    for i, (bin_id, (x1, y1, x2, y2), pixel_coords) in enumerate(detections):
        if i not in assigned_detections:
            mean_vx, mean_vy = estimate_flow(pixel_coords, res_x, res_y)
            new_projectile = Projectile((x1, y1, x2, y2), bin_id, (mean_vx, mean_vy))
            projectiles.append(new_projectile)
            #self.next_track_id += 1

    # Age and remove tracks that are missed too much
    for projectile in projectiles:
        if projectile not in assigned_projectiles:
            projectile.missed += 1
            if projectile.missed >= 3:
                projectile.confirmed = False

    


def draw_projectile_bboxes(frame, projectiles, res_mag,
                           num_bins: int = 36, debug: bool = False):
    """Render bounding boxes (+optional debug info) into <frame>."""
    # draw bounding boxes
    for p in projectiles:
        if not p.confirmed and not debug:
            continue
        x1, y1, x2, y2 = p.bbox
        cv.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        if debug:
            # angle bin label + mean magnitude
            start = p.bin_id * (360 / num_bins)
            end = (p.bin_id + 1) * (360 / num_bins)
            cv.putText(frame, f"{int(start)}-{int(end)}°",
                       (x1, y2 + 10), cv.FONT_HERSHEY_SIMPLEX,
                       0.4, (0, 255, 0), 1, cv.LINE_AA)
            roi_mag = res_mag[y1:y2, x1:x2]
            cv.putText(frame, f"{roi_mag.mean():.2f}",
                       (x1, y2 + 22), cv.FONT_HERSHEY_SIMPLEX,
                       0.4, (0, 255, 0), 1, cv.LINE_AA)

    # confirmed IDs on top
    for p in projectiles:
        if p.confirmed:
            x1, y1, _, _ = p.bbox
            label = f"ID {p.id}"
            (tw, th), _ = cv.getTextSize(label, cv.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv.rectangle(frame, (x1, y1 - 20), (x1 + tw, y1 - 5),
                         (255, 0, 0), cv.FILLED)
            cv.putText(frame, label, (x1, y1 - 7),
                       cv.FONT_HERSHEY_SIMPLEX, 0.5,
                       (255, 255, 255), 1, cv.LINE_AA)

## Initalize Video
In a real-time application this would run against a live camera feed. We create a mask at the center to not mistakenly identify our hero as a projectile. For enemies the same method would be used with the classifier above, but this is left as future work to implement.

In [ ]:
    
def display_frame(frame_number:int, return_frame:bool=False, center_mask:bool=False):
    cap = cv.VideoCapture(VID_PATH)
    for n in range(frame_number):
        cap.read()
    ret, first_frame = cap.read()
    assert ret, "Cannot read video"

    frame_down_sized = gaussian_downsize(first_frame)
    gray    = cv.cvtColor(frame_down_sized, cv.COLOR_BGR2GRAY)

    
    
    if center_mask:
        mask, (x1, y1, x2, y2) = create_center_mask(frame_down_sized.shape, mask_size_ratio=0.2, anchor_point=(295, 172))
        cv.rectangle(frame_down_sized, (x1, y1), (x2, y2), (255, 0, 0), 2)  
            
    plt.imshow(cv.cvtColor(frame_down_sized, cv.COLOR_BGR2RGB))
    plt.title("Downsized Frame" + (" with Center Mask" if center_mask else ""))
    plt.axis("off")
    plt.show()

    cap.release()

    if return_frame:
        return first_frame
    

display_frame(40, center_mask=True)

show_video_segment(VID_PATH, 0, 100, fps_override=5)



# Display optical field

In [ ]:

def plot_optical_flow(res_x, res_y):
    mag, angle = cv.cartToPolar(res_x, res_y)

    flow_image = np.zeros((res_x.shape[0], res_x.shape[1], 3), dtype=np.uint8)

    flow_image[..., 0] = (angle * 180 / np.pi / 2).astype(np.uint8)
    flow_image[..., 1] = 255
    flow_image[..., 2] = cv.normalize(mag, None, 0, 255, cv.NORM_MINMAX).astype(np.uint8)
    print(flow_image.shape)
    rgb = cv.cvtColor(flow_image, cv.COLOR_HSV2BGR)

    fig, axs = plt.subplots(1, 2, figsize=(14, 6))

    # Left: magnitude heatmap


    # Right: HSV visualization (direction + magnitude)
    axs[0].imshow(rgb)
    axs[0].set_title('Optical Flow (HSV-encoded)')
    axs[0].axis('off')

    im = axs[1].imshow(mag, cmap='hot')
    axs[1].set_title('Optical Flow Magnitude')
    axs[1].axis('off')
    fig.colorbar(im, ax=axs[1], shrink=0.8)

    plt.tight_layout()
    plt.show()


# Take two consecutive frames just to demo
def optical_flow(frame_number:int, display:bool):
    """Displays optical flow between the frame_number and the prior frame."""
    cap = cv.VideoCapture(VID_PATH)
    for n in range(frame_number-2):
        cap.read()
    _, frame1 = cap.read()
    cap.read()   
    cap.read()       
    _ , frame2 = cap.read()
    down_sized1 = gaussian_downsize(frame1)
    down_sized2 = gaussian_downsize(frame2)
    print(down_sized1.shape)

    gray1 = cv.cvtColor(down_sized1, cv.COLOR_BGR2GRAY)
    gray2 = cv.cvtColor(down_sized2, cv.COLOR_BGR2GRAY)

    flow = compute_optical_flow(gray1, gray2)
    if not display:
        return flow
    
    plot_optical_flow(flow[..., 0], flow[..., 1])
    
    return flow

flow = optical_flow(frame_number=40, display=True)

The color spesifies the angle of motion, the strength of the color is the magnitude of the motion. Blue dots are moving left and yellow dots are moving right.

### Subtracting Background Movement:
Fetching a frame sequence later in the video with background motion and subtracting the motion:

In [ ]:
frame_number = 2000

frame = display_frame(frame_number, return_frame=True)

flow = optical_flow(frame_number, display=True)

In [ ]:
res_x, res_y, bg_motion = subtract_background(flow)

plot_optical_flow(res_x, res_y)

In [ ]:
boxes = pixel_clustering(res_x, res_y, mag_threshold=0.5, size_threshold=10)
merged = filter_and_merge_bounding_boxes(boxes, min_l=10, max_l=100)

canvas = gaussian_downsize(frame).copy()
for _, (x1,y1,x2,y2), _ in merged:
    cv.rectangle(canvas, (x1,y1), (x2,y2), (0,255,0), 1)

plt.imshow(cv.cvtColor(canvas, cv.COLOR_BGR2RGB)); plt.axis("off")
plt.title("Merged candidate boxes")

We can see all projectiles are successfully captured, however there is a lot of background noise. This is handled with only confirming projectiles after 3 frames. THis will also filter out noise like the tourch. This is handled in the final pipeline which is built in the next cell.

In [ ]:
def run_pipeline(video_path=VID_PATH,
                 write_video=WRITE_VIDEO,
                 debug=DEBUG,
                 out_dir=OUT_DIR,
                 frame_sequence:tuple=None,
                 fps=10,
                 return_projectile_log:bool = False):
    
    '''Run projectile tracker over a sequence of frames.'''
    
    cap = cv.VideoCapture(video_path)
    fourcc = cv.VideoWriter_fourcc(*'avc1')
    writer = None
    projectiles:list[Projectile] = []
    next_id     = 0
    max_missed  = 3
    
    total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
    
    if frame_sequence is not None:
        start_frame, end_frame = frame_sequence
        assert 0 <= start_frame < total_frames, f"start_frame out of bounds: {start_frame}"
        assert start_frame < end_frame <= total_frames, f"Invalid end_frame: {end_frame}"
        cap.set(cv.CAP_PROP_POS_FRAMES, start_frame)
    else:
        start_frame, end_frame = 0, total_frames

    ret, prev_raw = cap.read()
    frame_h, frame_w = prev_raw.shape[:2]
    
    current_frame_idx = start_frame + 2
    
    if not ret: raise RuntimeError("Empty video")
    prev = gaussian_downsize(prev_raw)
    prev_gray = cv.cvtColor(prev, cv.COLOR_BGR2GRAY)

    
    if write_video:
        out_path = out_dir / (Path(video_path).stem + ".mp4")
        if os.path.exists(out_path):
            os.remove(out_path)
        writer   = cv.VideoWriter(str(out_path), fourcc, fps, (frame_w, frame_h))
    
    center_ignore_mask, _ = create_center_mask(prev.shape, mask_size_ratio=0.2, anchor_point=(295, 172))
    
    while cap.isOpened() and current_frame_idx < end_frame:
        ret, raw = cap.read()
        if not ret: 
            break
        
        #Skip identical frames
        max_pixel_diff = np.max(cv.absdiff(raw, prev_raw))
        if max_pixel_diff < 60:
            current_frame_idx += 1  # make sure you still increment
            continue
            
        
        fr   = gaussian_downsize(raw)
        gray = cv.cvtColor(fr, cv.COLOR_BGR2GRAY)

        flow         = compute_optical_flow(prev_gray, gray)
        
        flow[...,0] = flow[...,0] * center_ignore_mask
        flow[...,1] = flow[...,1] * center_ignore_mask
        
        
        
        res_x, res_y, bg_velocity = subtract_background(flow)
        boxes        = pixel_clustering(res_x, res_y, bg_velocity)
        merged       = filter_and_merge_bounding_boxes(boxes, 10, 100)

        # --- tracking state update (same logic as before) ---
        next_id = max([p.id for p in projectiles if p.id is not None], default=0) + 1
        update_projectiles(projectiles, merged, res_x, res_y,
                           next_id=next_id, bg_velocity=bg_velocity) 
        

        # --- draw ---
        draw_frame = fr.copy()
        draw_projectile_bboxes(draw_frame, projectiles, res_x, res_y, debug)

        up = cv.resize(draw_frame, (frame_w,
                                    frame_h), interpolation=cv.INTER_LINEAR)
        
        if write_video: 
            assert up.shape[1] == frame_w 
            assert up.shape[0] == frame_h 
            writer.write(up)
        
        if debug:
            cv.imshow("debug", up)
            if cv.waitKey(1) & 0xFF == ord('q'): break

        prev_gray = gray
        prev_raw = raw

    cap.release()
    if write_video: 
        writer.release()
    cv.destroyAllWindows()
    projectile_info_for_predictor = [(projectile.id, projectile.center, projectile.global_velocity_log) for projectile in projectiles if projectile.confirmed]
    
    return str(out_path) if write_video else None, projectile_info_for_predictor if return_projectile_log else None

#Return projectile holding information about projectile movement and current position for predictor.
video_path, projectile_log_for_predictor = run_pipeline(frame_sequence=(2000, 2100), return_projectile_log=True)




In [ ]:
print(video_path)



show_video_segment(video_path)



In [ ]:
# Format of projectile_log_for_predictor: [(projectile.id, projectile.center, projectile.global_velocity_log)]

print(projectile_log_for_predictor[0])

print(len(projectile_log_for_predictor[0][1]))

# Inspect and run other videos 

Run videos from projectile_tracking/120fps_data folder:

In [ ]:
start_frame = 900
end_frame = 1500
path = 'projectile_tracking/120fps_data/different_projectiles_and_moving.mp4'


show_video_segment(path, start_frame, end_frame)


In [ ]:
video_path, projectile_log_for_predictor2 = run_pipeline(video_path=path, frame_sequence=(start_frame, 1200), fps=30, return_projectile_log=True)

In [ ]:
show_video_segment(video_path)

from moviepy.editor import VideoFileClip

# Load the video
clip = VideoFileClip(str(video_path))

# (Optional) Trim a segment
gif_clip = clip.subclip(0, 5)  # first 2 seconds

# Resize for presentation (optional)
gif_clip = gif_clip.resize(width=800)

# Export to GIF
gif_clip.write_gif("projectile_demo3.gif", fps=30)

We have now are now effectivly tracking pixels even with a moving background.

## Projectile Prediction
This sections shows how the projectile predictor can be used to predict future positions given a history.

In [ ]:
from projectile_prediction.predictor_new import predict_future

In [ ]:
# Version 1: Horizontal Straight Line
def generate_straight_trajectory_v1(t, start_pos=(0, 0), velocity=(200, 0)):
    """Generate position at time t for straight-line movement"""
    x = start_pos[0] + velocity[0] * t
    y = start_pos[1] + velocity[1] * t
    return (x, y)

# Version 2: Angled Straight Line
def generate_straight_trajectory_v2(t, start_pos=(0, 0), velocity=(30, 30)):
    """Generate position at time t for straight-line movement at 45 degrees"""
    x = start_pos[0] + velocity[0] * t
    y = start_pos[1] + velocity[1] * t
    return (x, y)

# Circle Trajectory
def generate_circle_trajectory(t, center=(200, 200), radius=100, angular_speed=1.0, phase=0):
    """Generate position at time t for circular movement"""
    angle = angular_speed * t + phase
    x = center[0] + radius * np.cos(angle)
    y = center[1] + radius * np.sin(angle)
    return (x, y)

# Version 1: Horizontal Sine Wave
def generate_sine_trajectory_v1(t, start_pos=(0, 0), x_velocity=200, amplitude=50, frequency=2, phase=0):
    """Generate position at time t for sinusoidal movement (horizontal)"""
    x = start_pos[0] + x_velocity * t
    # Simple sine wave based on x position (adjust divisor as needed for wavelength)
    # The divisor controls how many spatial units correspond to one cycle
    y = start_pos[1] + amplitude * np.sin(2 * np.pi * frequency * x / (x_velocity * 1.5) + phase) 
    return (x, y)

# Version 2: Angled Sine Wave
def generate_sine_trajectory_v2(t, start_pos=(0, 0), angle_deg=-45, speed=40, amplitude=50, frequency=2, phase=0):
    """Generate position at time t for sinusoidal movement at an angle"""
    angle_rad = np.radians(angle_deg)
    base_x = start_pos[0] + speed * t * np.cos(angle_rad)
    base_y = start_pos[1] + speed * t * np.sin(angle_rad)
    perp_angle = angle_rad + np.pi/2
    x = base_x + amplitude * np.sin(2 * np.pi * frequency * t + phase) * np.cos(perp_angle)
    y = base_y + amplitude * np.sin(2 * np.pi * frequency * t + phase) * np.sin(perp_angle)
    return (x, y)

def add_noise(positions, noise_level=2.0):
    """Add Gaussian noise to a list of positions"""
    noisy_positions = []
    for pos in positions:
        noise_x = np.random.normal(0, noise_level)
        noise_y = np.random.normal(0, noise_level)
        noisy_positions.append((pos[0] + noise_x, pos[1] + noise_y))
    return noisy_positions

Simple function for simulating, predicting, and plotting.

In [ ]:
def run_and_plot_simulation(dt, n_steps, future_steps, noise_level, trajectory_generators, figure_title, seed=0, plot_clean_history=False):
    num_plots = len(trajectory_generators)
    fig, axes = plt.subplots(num_plots, 1, figsize=(8, 4 * num_plots))
    fig.suptitle(figure_title, fontsize=16)
    np.random.seed(seed)
    for i, (name, trajectory_func) in enumerate(trajectory_generators):
        ax = axes[i]
        print(f"Processing: {name}")

        # Generate trajectory data
        total_steps = n_steps + future_steps
        all_times = np.arange(0, total_steps * dt, dt)
        # Ensure enough points are generated
        if len(all_times) < total_steps:
             all_times = np.arange(0, (total_steps + 1) * dt, dt)[:total_steps]

        all_positions = [trajectory_func(t) for t in all_times]

        history_times = all_times[:n_steps]
        history_pos_clean = all_positions[:n_steps]
        actual_future_times = all_times[n_steps:total_steps]
        actual_future_pos = all_positions[n_steps:total_steps]

        # Add noise
        current_noise = noise_level
        if name == "Straight (Horizontal)": 
            # Less noise for straight line, this is just done because the straight line is really sensitive to the noise because of the short distance it travels in the examples.
            current_noise = noise_level / 3 if noise_level > 0 else 0 
            ax.set_ylim(-10, 10) # Zoom in for straight line

        history_pos_noisy = add_noise(history_pos_clean, current_noise)

        # Prepare history for predictor
        time_position_history = [[t, pos[0], pos[1]] for t, pos in zip(history_times, history_pos_noisy)]

        # Predict future
        predicted_future = predict_future(
            time_position_history, 
            future_times=actual_future_times,
            num_points=future_steps
        )

        # Convert to numpy arrays for plotting
        history_np_clean = np.array(history_pos_clean)
        history_np_noisy = np.array(history_pos_noisy)
        actual_future_np = np.array(actual_future_pos)
        # Ensure predicted_future is not empty before converting
        predicted_future_np = np.array(predicted_future) if predicted_future else np.empty((0,2))

        # --- Plotting ---
        # Plot actual future
        if actual_future_np.shape[0] > 0:
             ax.plot(actual_future_np[:, 0], actual_future_np[:, 1], 'g-', label="Actual Future", linewidth=2)
        
        # Plot predicted future if available
        if predicted_future_np.shape[0] > 0:
            ax.plot(predicted_future_np[:, 0], predicted_future_np[:, 1], 'r--', label="Predicted Future", linewidth=2)

        # Plot noisy history (input)
        if history_np_noisy.shape[0] > 0:
             ax.plot(history_np_noisy[:, 0], history_np_noisy[:, 1], 'bo', markersize=4, label="Input Points")

        # Plot clean history (ground truth)
        if history_np_clean.shape[0] > 0 and plot_clean_history:
             ax.plot(history_np_clean[:, 0], history_np_clean[:, 1], 'k--', label="Clean History", linewidth=2)
        
        # Add titles and labels
        ax.set_title(f"{name} - Noise σ: {current_noise:.2f} px")
        ax.set_xlabel("x position (pixels)")
        ax.set_ylabel("y position (pixels)")
        ax.legend()
        ax.grid(True)
        
        # Set aspect ratio for circle or rotated plots
        if "Circle" in name or "°" in name:
            ax.set_aspect('equal', adjustable='datalim')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
    plt.show()

#### Simulated Conditions - No Noise

In [ ]:
# Simulation parameters
dt_example = 0.05
n_steps_example = 20 # Total 1 second of history
future_steps_example = 20
noise_level_example = 0.0 # Example: No noise

# Trajectory generators 
trajectory_generators_example = [
    ("Straight", lambda t: generate_straight_trajectory_v1(t, start_pos=(0, 0), velocity=(200, 0))),
    ("Circle", lambda t: generate_circle_trajectory(t, center=(200, 200), radius=100, angular_speed=np.pi)),
    ("Sine", lambda t: generate_sine_trajectory_v1(t, start_pos=(0, 0), x_velocity=200, amplitude=50, frequency=2))
]

# Title for the plot
figure_title_example = ''

# Call the function
run_and_plot_simulation(
    dt=dt_example,
    n_steps=n_steps_example,
    future_steps=future_steps_example,
    noise_level=noise_level_example,
    trajectory_generators=trajectory_generators_example,
    figure_title=figure_title_example
)

#### Rotated with no noise, just to show that it works with movement that does not just go horizonatally

In [ ]:
dt_v2 = 0.05
n_steps_v2 = 20
future_steps_v2 = 20
noise_level_v2 = 0.0

trajectory_generators_v2 = [
    ("Straight (45°)", lambda t: generate_straight_trajectory_v2(t, start_pos=(0, 0), velocity=(30, 30))),
    # Using different circle parameters for variety
    ("Circle", lambda t: generate_circle_trajectory(t, center=(150, 150), radius=80, angular_speed=np.pi)), 
    ("Sine (-90°)", lambda t: generate_sine_trajectory_v2(t, start_pos=(0, 0), angle_deg=-90, speed=200, amplitude=50, frequency=2))
]

# Create figure
fig2, axes2 = plt.subplots(len(trajectory_generators_v2), 1, figsize=(8, 12))
# fig2.suptitle('Demonstrating Rotated Trajectories', fontsize=16)

np.random.seed(0)

for i, (name, trajectory_func) in enumerate(trajectory_generators_v2):
    ax = axes2[i]
    print(f"Processing: {name}")

    # Generate trajectory data
    total_steps = n_steps_v2 + future_steps_v2
    all_times = np.arange(0, total_steps * dt_v2, dt_v2)
    all_positions = [trajectory_func(t) for t in all_times]

    history_times = all_times[:n_steps_v2]
    history_pos_clean = all_positions[:n_steps_v2]
    actual_future_times = all_times[n_steps_v2:total_steps]
    actual_future_pos = all_positions[n_steps_v2:total_steps]

    # Add noise
    current_noise = noise_level_v2
    if "Straight" in name:
        # Less noise for straight line, this is just done because the straight line is really sensitive to the noise because of the short distance is traveles in the examples.
        current_noise = noise_level_v2 / 3 
    history_pos_noisy = add_noise(history_pos_clean, current_noise)

    # Prepare history for predictor
    time_position_history = [[t, pos[0], pos[1]] for t, pos in zip(history_times, history_pos_noisy)]

    # Predict future
    predicted_future = predict_future(
        time_position_history, 
        future_times=actual_future_times,
        num_points=future_steps_v2
    )

    # Convert to numpy arrays for plotting
    history_np_clean = np.array(history_pos_clean)
    history_np_noisy = np.array(history_pos_noisy)
    actual_future_np = np.array(actual_future_pos)
    # Ensure predicted_future is not empty before converting
    predicted_future_np = np.array(predicted_future) if predicted_future else np.empty((0,2))

    # --- Plotting ---
    # Plot actual future (ground truth)
    ax.plot(actual_future_np[:, 0], actual_future_np[:, 1], 'g-', label="Actual Future", linewidth=2)
    
    # Plot predicted future if available
    if predicted_future_np.shape[0] > 0:
        ax.plot(predicted_future_np[:, 0], predicted_future_np[:, 1], 'r--', label="Predicted Future", linewidth=2)
    
    # Plot clean history (faded)
    # ax.plot(history_np_clean[:, 0], history_np_clean[:, 1], 'c.', alpha=0.5, label="True Historical Path")
    
    # Plot noisy history (input)
    ax.plot(history_np_noisy[:, 0], history_np_noisy[:, 1], 'bo', markersize=4, label="Input Points")

    ax.set_title(f"{name} - Noise σ: {current_noise:.2f} px")
    ax.set_xlabel("x position (pixels)")
    ax.set_ylabel("y position (pixels)")
    ax.legend()
    ax.grid(True)
    
    ax.set_aspect('equal', adjustable='datalim')

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

#### With some noise added to the data

In [ ]:
# Simulation parameters
dt_example = 0.05
n_steps_example = 20
future_steps_example = 20
noise_level_example = 3.0

# Trajectory generators 
trajectory_generators_example = [
    ("Straight (Horizontal)", lambda t: generate_straight_trajectory_v1(t, start_pos=(0, 0), velocity=(200, 0))),
    ("Circle", lambda t: generate_circle_trajectory(t, center=(200, 200), radius=100, angular_speed=np.pi)),
    ("Sine (Horizontal)", lambda t: generate_sine_trajectory_v1(t, start_pos=(0, 0), x_velocity=200, amplitude=50, frequency=2))
]

figure_title_example = ''

# Call the function
run_and_plot_simulation(
    dt=dt_example,
    n_steps=n_steps_example,
    future_steps=future_steps_example,
    noise_level=noise_level_example,
    trajectory_generators=trajectory_generators_example,
    figure_title=figure_title_example
)

#### With allot of noise

In [ ]:
# Simulation parameters
dt_example = 0.05
n_steps_example = 20
future_steps_example = 20
noise_level_example = 5

# Trajectory generators 
trajectory_generators_example = [
    ("Straight (Horizontal)", lambda t: generate_straight_trajectory_v1(t, start_pos=(0, 0), velocity=(200, 0))),
    ("Circle", lambda t: generate_circle_trajectory(t, center=(200, 200), radius=100, angular_speed=3.0)),
    ("Sine (Horizontal)", lambda t: generate_sine_trajectory_v1(t, start_pos=(0, 0), x_velocity=200, amplitude=50, frequency=2))
]

# Title for the plot
figure_title_example = ''

# Call the function
run_and_plot_simulation(
    dt=dt_example,
    n_steps=n_steps_example,
    future_steps=future_steps_example,
    noise_level=noise_level_example,
    trajectory_generators=trajectory_generators_example,
    figure_title=figure_title_example
)